In [21]:
import os
import re
import glob
import pandas as pd
import numpy as np

# =========================
# 配置区：改这里就行
# =========================
ROOT_DIR = "idata/现券成交分机构统计/"   # 你的889个excel目录
PATTERN  = "现券成交分机构统计*.xlsx"       # 文件名模式
OUT_PARQUET = "idata/现券成交分机构/bond_trades_all_in_one.parquet"
OUT_CSV     = "idata/现券成交分机构/bond_trades_all_in_one.csv"

# sheet 名（你给的文件就是这三个）
SHEET_NET  = "机构净买入债券成交金额统计表"
SHEET_BUY  = "机构买入债券成交金额统计表"
SHEET_SELL = "机构卖出债券成交金额统计表"

# 表头行（你之前文件 header=3 对齐“机构类型/期限/国债-新债...”）
HEADER_ROW = 3

# =========================
# 工具函数
# =========================
def extract_date_from_filename(fp: str) -> pd.Timestamp:
    """
    从文件名提取日期：优先取第一个8位数字（YYYYMMDD）
    """
    base = os.path.basename(fp)
    m = re.search(r"(\d{8})", base)
    if not m:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")

def read_one_sheet(fp: str, sheet_name: str) -> pd.DataFrame:
    """
    读取单个 sheet 并清洗成宽表：
    columns = [机构类型, 期限, ...各资产类别列...]
    """
    df = pd.read_excel(fp, sheet_name=sheet_name, header=HEADER_ROW)

    # 防御：有时会多出全空行/列
    df = df.dropna(how="all")
    df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]

    # 标准化前两列列名
    cols = list(df.columns)
    if len(cols) < 2:
        raise ValueError(f"{fp} / {sheet_name}: 列数不足，无法解析")
    cols[0] = "机构类型"
    cols[1] = "期限"
    df.columns = cols

    # 清洗文本
    df["机构类型"] = df["机构类型"].astype(str).str.strip()
    df["期限"] = df["期限"].astype(str).str.strip()

    # 数值列：把 -- 等替换为0，再转float
    for c in df.columns[2:]:
        df[c] = (
            df[c]
            .replace({"--": 0, "—": 0, "-": 0})
            .replace({np.nan: 0})
        )
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    return df

def wide_to_long(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    """
    宽表 -> 长表：每行=机构类型×期限×资产类别×value
    """
    id_vars = ["机构类型", "期限"]
    value_vars = [c for c in df.columns if c not in id_vars]
    out = df.melt(id_vars=id_vars, value_vars=value_vars,
                  var_name="资产类别", value_name=value_name)
    return out

def parse_one_file(fp: str) -> pd.DataFrame:
    """
    读取一个excel文件，输出统一长表：
    日期, 机构类型, 期限, 资产类别, Buy, Sell, Net, Turnover
    """
    asof = extract_date_from_filename(fp)

    net_w = read_one_sheet(fp, SHEET_NET)
    buy_w = read_one_sheet(fp, SHEET_BUY)
    sell_w = read_one_sheet(fp, SHEET_SELL)

    net_l = wide_to_long(net_w, "Net")
    buy_l = wide_to_long(buy_w, "Buy")
    sell_l = wide_to_long(sell_w, "Sell")

    # 合并
    m = net_l.merge(buy_l, on=["机构类型", "期限", "资产类别"], how="outer") \
             .merge(sell_l, on=["机构类型", "期限", "资产类别"], how="outer")

    # 缺失补0
    for c in ["Net", "Buy", "Sell"]:
        if c not in m.columns:
            m[c] = 0.0
        m[c] = pd.to_numeric(m[c], errors="coerce").fillna(0.0)

    m["Turnover"] = m["Buy"] + m["Sell"]
    m.insert(0, "日期", asof)

    # 可选：过滤掉“合计”这类资产类别（如果你只要明细）
    # m = m[m["资产类别"] != "合计"]

    return m

# =========================
# 主流程：批量整合
# =========================
def build_all():
    files = sorted(glob.glob(os.path.join(ROOT_DIR, PATTERN)))
    if not files:
        raise FileNotFoundError(f"未找到文件：{os.path.join(ROOT_DIR, PATTERN)}")

    all_parts = []
    bad = []

    for i, fp in enumerate(files, 1):
        try:
            part = parse_one_file(fp)
            all_parts.append(part)
        except Exception as e:
            bad.append((fp, str(e)))
            print(f"[WARN] 解析失败 ({i}/{len(files)}): {os.path.basename(fp)} -> {e}")

        if i % 50 == 0:
            print(f"[INFO] 已处理 {i}/{len(files)}")

    out = pd.concat(all_parts, ignore_index=True)
    out = out.sort_values(["日期", "机构类型", "期限", "资产类别"]).reset_index(drop=True)

    # 输出：Parquet（推荐，快且省空间）+ CSV（通用）
    os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)
    out.to_parquet(OUT_PARQUET, index=False)
    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[DONE] 合并完成：{len(files)} 个文件 -> {len(out):,} 行")
    print(f"[DONE] Parquet: {OUT_PARQUET}")
    print(f"[DONE] CSV:     {OUT_CSV}")

    if bad:
        bad_df = pd.DataFrame(bad, columns=["file", "error"])
        bad_path = os.path.join(os.path.dirname(OUT_PARQUET), "bad_files.csv")
        bad_df.to_csv(bad_path, index=False, encoding="utf-8-sig")
        print(f"[WARN] 有 {len(bad)} 个文件失败，详见：{bad_path}")

    return out

if __name__ == "__main__":
    build_all()

/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

[INFO] 已处理 50/892


/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

[INFO] 已处理 100/892


/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

...

[INFO] 已处理 700/892
[INFO] 已处理 750/892


/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

[INFO] 已处理 800/892
[INFO] 已处理 850/892


/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"--": 0, "—": 0, "-": 0})
/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/3983752021.py:63: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old b

[DONE] 合并完成：892 个文件 -> 1,253,448 行
[DONE] Parquet: idata/现券成交分机构/all_in_one.parquet
[DONE] CSV:     idata/现券成交分机构/all_in_one.csv


In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 配置
# =========================
CSV_PATH = Path("/Users/xianqu/Documents/金市/2026Q1/机构行为/idata/现券成交分机构/bond_trades_all_in_one.csv")
OUT_PARQUET = Path("mvp_factors.parquet")
OUT_CSV = Path("mvp_factors.csv")

# 输出增强版（含zscore/EWMA/三联评分）
OUT_PARQUET_ENH = Path("mvp_factors_enhanced.parquet")
OUT_CSV_ENH = Path("mvp_factors_enhanced.csv")

# zscore / EWMA 参数
ZSCORE_WIN = 60
EWMA_HALFLIFE_10 = 10   # 约等价“10日尺度”
EWMA_HALFLIFE_20 = 20   # 约等价“20日尺度”

# 只做利率债（你们FVOCI关心的）
RATE_ASSETS = {
    "国债-新债", "国债-老债",
    "政策性金融债-新债", "政策性金融债-老债",
    "地方政府债"
}

# 期限 -> 近似久期权重（可按你们内部口径微调）
TENOR_DUR_W = {
    "≦1Y": 0.5, "≤1Y": 0.5, "<=1Y": 0.5, "1Y以下": 0.5,
    "1-3Y": 2.0,
    "3-5Y": 4.0,
    "5-7Y": 6.0,
    "7-10Y": 8.5,
    "10Y+": 12.0, "10Y以上": 12.0, "10年以上": 12.0, "10Y及以上": 12.0
}

LONG_BUCKETS = {"7-10Y", "10Y+", "10Y以上", "10年以上", "10Y及以上", "10-15Y", "15Y+"}
SHORT_BUCKETS = {"≦1Y", "≤1Y", "<=1Y", "1Y以下", "1-3Y"}  # 若你要短端=≤1Y，把"1-3Y"删掉


# =========================
# 读入与清洗（兼容两种schema）
# =========================
def load_any_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    has_a = {"日期", "机构类型", "期限", "资产类别"}.issubset(df.columns) and {"Net"}.issubset(df.columns)
    has_b = {"日期", "机构类型", "期限", "资产类别", "成交净额"}.issubset(df.columns)

    if not (has_a or has_b):
        raise ValueError(
            "CSV 列名不符合预期。\n"
            f"当前列名：{df.columns.tolist()}\n"
            "期望 schema A: 日期,机构类型,期限,资产类别,Net,Buy,Sell,Turnover\n"
            "或 schema B: 日期,机构类型,期限,资产类别,成交净额"
        )

    for c in ["机构类型", "期限", "资产类别"]:
        df[c] = df[c].astype(str).str.strip()

    # 日期解析（兼容 2022-06-27 / 20220627）
    df["日期"] = pd.to_datetime(df["日期"].astype(str).str.strip(), errors="coerce")
    df = df.dropna(subset=["日期"])

    def to_num(s: pd.Series) -> pd.Series:
        s = s.astype(str).str.strip()
        s = s.replace({"--": "0", "—": "0", "-": "0", "nan": "0", "None": "0"})
        return pd.to_numeric(s, errors="coerce").fillna(0.0)

    if has_a:
        for c in ["Net", "Buy", "Sell", "Turnover"]:
            if c in df.columns:
                df[c] = to_num(df[c])
        if "Turnover" not in df.columns:
            df["Turnover"] = df.get("Buy", 0.0) + df.get("Sell", 0.0)
    else:
        df["Net"] = to_num(df["成交净额"])
        df["Buy"] = 0.0
        df["Sell"] = 0.0
        df["Turnover"] = np.nan

    return df


def add_helpers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["DurW"] = df["期限"].map(TENOR_DUR_W).fillna(0.0)
    df["is_long"] = df["期限"].isin(LONG_BUCKETS)
    df["is_short"] = df["期限"].isin(SHORT_BUCKETS)

    def sector(asset: str) -> str:
        if asset.startswith("国债"):
            return "国债"
        if asset.startswith("政策性金融债"):
            return "政金"
        if asset == "地方政府债":
            return "地方"
        return "其他"

    df["Sector"] = df["资产类别"].map(sector)
    return df


# =========================
# 因子计算（单日）
# =========================
def compute_daily_factors(d: pd.DataFrame) -> pd.Series:
    # 去掉“合计”行（避免重复计数）
    d = d[d["资产类别"] != "合计"].copy()

    # 1) 长端净买入（7Y+）
    net_long = d.loc[d["is_long"], "Net"].sum()

    # 2) 长端净买入强度
    if d["Turnover"].notna().any() and d["Turnover"].sum() > 0:
        turn_long = d.loc[d["is_long"], "Turnover"].sum()
        f2 = net_long / (turn_long + 1e-12)
        turnover_total = d["Turnover"].sum()
    else:
        abs_long = d.loc[d["is_long"], "Net"].abs().sum()
        f2 = net_long / (abs_long + 1e-12)
        turnover_total = np.nan

    # 3) 久期倾向 DurFlow
    f3 = (d["Net"] * d["DurW"]).sum()

    # 4) 曲线偏好：长端 - 短端
    net_short = d.loc[d["is_short"], "Net"].sum()
    f4 = net_long - net_short

    # 5) 政金相对国债
    gov_net = d.loc[d["Sector"] == "国债", "Net"].sum()
    pfb_net = d.loc[d["Sector"] == "政金", "Net"].sum()
    lg_net = d.loc[d["Sector"] == "地方", "Net"].sum()
    f5 = pfb_net - gov_net

    # 6) 地方相对政金
    f6 = lg_net - pfb_net

    # 7) 国债新老偏好
    gov_new = d.loc[d["资产类别"] == "国债-新债", "Net"].sum()
    gov_old = d.loc[d["资产类别"] == "国债-老债", "Net"].sum()
    f7 = gov_new - gov_old

    # 8) 政金新老偏好
    pfb_new = d.loc[d["资产类别"] == "政策性金融债-新债", "Net"].sum()
    pfb_old = d.loc[d["资产类别"] == "政策性金融债-老债", "Net"].sum()
    f8 = pfb_new - pfb_old

    # 9/10) 拥挤×集中度（跨机构，基于利率债总Net）
    inst = d.groupby("机构类型")["Net"].sum()
    abs_sum = inst.abs().sum() + 1e-12
    crowd_buy = inst.clip(lower=0).sum() / abs_sum
    crowd_sell = (-inst).clip(lower=0).sum() / abs_sum
    top3_share = inst.abs().sort_values(ascending=False).head(3).sum() / abs_sum
    f9 = crowd_buy * top3_share
    f10 = crowd_sell * top3_share

    return pd.Series({
        "F1_长端净买入(7Y+)_亿元": net_long,
        "F2_长端净买入强度(Net/Turnover或代理)": f2,
        "F3_久期倾向DurFlow(净买入×久期权重)": f3,
        "F4_曲线偏好CurveFlow(长端-短端净买入)_亿元": f4,
        "F5_政金相对国债偏好(PFB-GOV)_亿元": f5,
        "F6_地方相对政金偏好(LG-PFB)_亿元": f6,
        "F7_国债新老偏好(新-老)_亿元": f7,
        "F8_政金新老偏好(新-老)_亿元": f8,
        "F9_买方拥挤×集中度": f9,
        "F10_卖方拥挤×集中度": f10,
        "CrowdBuy": crowd_buy,
        "CrowdSell": crowd_sell,
        "Top3_share_|Net|": top3_share,
        "净买入合计(利率债)_亿元": d["Net"].sum(),
        "成交合计(利率债,Turnover)_亿元": turnover_total,
    })


# =========================
# 指标增强：zscore / EWMA / 三联评分
# =========================
def rolling_zscore(s: pd.Series, win: int) -> pd.Series:
    m = s.rolling(win, min_periods=max(10, win // 3)).mean()
    sd = s.rolling(win, min_periods=max(10, win // 3)).std(ddof=0)
    return (s - m) / (sd + 1e-12)

def add_enhancements(factors: pd.DataFrame) -> pd.DataFrame:
    out = factors.copy()
    out = out.sort_values("日期").reset_index(drop=True)

    core_cols = [
        "F1_长端净买入(7Y+)_亿元",
        "F2_长端净买入强度(Net/Turnover或代理)",
        "F3_久期倾向DurFlow(净买入×久期权重)",
        "F4_曲线偏好CurveFlow(长端-短端净买入)_亿元",
        "F5_政金相对国债偏好(PFB-GOV)_亿元",
        "F6_地方相对政金偏好(LG-PFB)_亿元",
        "F7_国债新老偏好(新-老)_亿元",
        "F8_政金新老偏好(新-老)_亿元",
        "F9_买方拥挤×集中度",
        "F10_卖方拥挤×集中度",
    ]

    # zscore60 / ewma10 / ewma20
    for c in core_cols:
        out[f"{c}_z{ZSCORE_WIN}"] = rolling_zscore(out[c], ZSCORE_WIN)
        out[f"{c}_ewmHL{EWMA_HALFLIFE_10}"] = out[c].ewm(halflife=EWMA_HALFLIFE_10, adjust=False).mean()
        out[f"{c}_ewmHL{EWMA_HALFLIFE_20}"] = out[c].ewm(halflife=EWMA_HALFLIFE_20, adjust=False).mean()

    # 三联评分（用zscore版本做合成更稳）
    zF3 = out[f"F3_久期倾向DurFlow(净买入×久期权重)_z{ZSCORE_WIN}"]
    zF4 = out[f"F4_曲线偏好CurveFlow(长端-短端净买入)_亿元_z{ZSCORE_WIN}"]
    zF5 = out[f"F5_政金相对国债偏好(PFB-GOV)_亿元_z{ZSCORE_WIN}"]
    zF6 = out[f"F6_地方相对政金偏好(LG-PFB)_亿元_z{ZSCORE_WIN}"]
    zF9 = out[f"F9_买方拥挤×集中度_z{ZSCORE_WIN}"]
    zF10 = out[f"F10_卖方拥挤×集中度_z{ZSCORE_WIN}"]

    # A：久期开关评分（加久期倾向）：F3为主，F4辅助（长端更强则更倾向加久期）
    out["Score_A_久期(加久期倾向)"] = 0.7 * zF3 + 0.3 * zF4

    # B：券种RV评分：政金相对国债 + 地方相对政金
    out["Score_B_RV(券种轮动)"] = 0.5 * zF5 + 0.5 * zF6

    # C：执行风险/拥挤评分：买方拥挤为风险上升；卖方拥挤（被动卖压）更多是“可接货”
    # 这里定义为“追涨风险”：买拥挤加分，卖拥挤减分
    out["Score_C_执行(追涨风险)"] = 0.6 * zF9 - 0.4 * zF10

    # 信号灯（-1/0/+1）阈值可调：|score|>0.5 视为显著
    def tri_signal(x, th=0.5):
        return np.where(x > th, 1, np.where(x < -th, -1, 0))

    out["Signal_A_久期"] = tri_signal(out["Score_A_久期(加久期倾向)"])
    out["Signal_B_RV"] = tri_signal(out["Score_B_RV(券种轮动)"])
    out["Signal_C_执行"] = tri_signal(out["Score_C_执行(追涨风险)"])

    return out


# =========================
# 主函数：生成日序列并保存
# =========================
def build_mvp_factors(csv_path: Path) -> pd.DataFrame:
    df = load_any_csv(csv_path)

    # 只保留利率债（注意：这里不再保留“合计”，因为单日计算里已经会剔除合计）
    df = df[df["资产类别"].isin(RATE_ASSETS) | (df["资产类别"] == "合计")].copy()
    df = add_helpers(df)

    factors = (
        df.groupby("日期", sort=True)
        .apply(compute_daily_factors)
        .reset_index()
        .sort_values("日期")
        .reset_index(drop=True)
    )

    # 基础输出
    factors.to_parquet(OUT_PARQUET, index=False)
    factors.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[DONE] 基础输出：{OUT_PARQUET} / {OUT_CSV}")
    print(f"日期范围：{factors['日期'].min().date()} ~ {factors['日期'].max().date()}，共 {len(factors)} 天")

    # 增强输出
    enh = add_enhancements(factors)
    enh.to_parquet(OUT_PARQUET_ENH, index=False)
    enh.to_csv(OUT_CSV_ENH, index=False, encoding="utf-8-sig")
    print(f"[DONE] 增强输出：{OUT_PARQUET_ENH} / {OUT_CSV_ENH}")

    return enh


if __name__ == "__main__":
    build_mvp_factors(CSV_PATH)

[DONE] 基础输出：mvp_factors.parquet / mvp_factors.csv
日期范围：2022-06-27 ~ 2026-03-06，共 883 天
[DONE] 增强输出：mvp_factors_enhanced.parquet / mvp_factors_enhanced.csv


/var/folders/61/b8tn4bp939b8r_181vlgq1r40000gn/T/ipykernel_16862/2684923204.py:252: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_daily_factors)


In [ ]:
df.tail()

In [20]:
df['机构类型'].value_counts()

机构类型
证券公司            103680
保险公司            103680
理财子公司及理财类产品     103680
货币市场基金          103680
基金公司及产品         103680
其他              103680
大型商业银行/政策性银行     99360
股份制商业银行          99360
城市商业银行           99360
外资银行             99360
农村金融机构           99360
其他产品类            99360
大型银行              4320
中小型银行             4320
理财类产品             3720
Name: count, dtype: int64